In [3]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import requests
import ee

In [4]:
ee.Authenticate()
ee.Initialize(project='geohan-solar')
print("GEE hazır!")

GEE hazır!


In [5]:
veri = rasterio.open("n38_e032_1arc_v3.tif")
yukseklik = veri.read(1)
print("Veri yüklendi!")
print("Boyut:", veri.width, "x", veri.height, "piksel")

Veri yüklendi!
Boyut: 3601 x 3601 piksel


In [6]:
piksel_metre = 30
x_egim = np.gradient(yukseklik, piksel_metre, axis=1)
y_egim = np.gradient(yukseklik, piksel_metre, axis=0)
egim = np.sqrt(x_egim**2 + y_egim**2)
egim_derece = np.degrees(np.arctan(egim))
print("Ortalama eğim:", egim_derece.mean().round(2), "derece")

Ortalama eğim: 3.49 derece


In [7]:
lat = 38.5
lon = 32.5

url = f"https://power.larc.nasa.gov/api/temporal/climatology/point?parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}&latitude={lat}&format=JSON"

response = requests.get(url)
veri_ghi = response.json()
ghi_aylik = veri_ghi['properties']['parameter']['ALLSKY_SFC_SW_DWN']
yillik_ghi = ghi_aylik['ANN'] * 365

print(f"Yıllık GHI: {yillik_ghi:.0f} kWh/m²/yıl")

Yıllık GHI: 1750 kWh/m²/yıl


In [8]:
lat = 38.5
lon = 32.5

url = f"https://power.larc.nasa.gov/api/temporal/climatology/point?parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}&latitude={lat}&format=JSON"

response = requests.get(url)
veri_ghi = response.json()
ghi_aylik = veri_ghi['properties']['parameter']['ALLSKY_SFC_SW_DWN']
yillik_ghi = ghi_aylik['ANN'] * 365

print(f"Yıllık GHI: {yillik_ghi:.0f} kWh/m²/yıl")

Yıllık GHI: 1750 kWh/m²/yıl


In [9]:
egim_skoru = 100 if egim_derece.mean() < 5 else 70

if yillik_ghi > 1800:
    ghi_skoru = 100
elif yillik_ghi > 1600:
    ghi_skoru = 80
elif yillik_ghi > 1400:
    ghi_skoru = 60
else:
    ghi_skoru = 40

solar_skor = (egim_skoru * 0.40) + (ghi_skoru * 0.60)

print("=== SOLAR-INTELLIGENCE RAPORU ===")
print(f"Bölge: Konya-Ankara")
print(f"Eğim skoru:  {egim_skoru}/100")
print(f"GHI skoru:   {ghi_skoru}/100")
print(f"TOPLAM SKOR: {solar_skor:.0f}/100")

=== SOLAR-INTELLIGENCE RAPORU ===
Bölge: Konya-Ankara
Eğim skoru:  100/100
GHI skoru:   80/100
TOPLAM SKOR: 88/100


In [11]:
arazi_hektar = 100
kurulu_guc_mw = arazi_hektar * 0.5
uretim_kwh = yillik_ghi * kurulu_guc_mw * 1000 * 0.80

# Güncel fiyatlar
kwh_fiyat_tl = 4.20        # 2024 lisanssız satış fiyatı TL/kWh
guncel_kur = 38.0           # USD/TL

gelir_tl = uretim_kwh * kwh_fiyat_tl
yatirim_dolar = kurulu_guc_mw * 1_200_000
yatirim_tl = yatirim_dolar * guncel_kur
geri_odeme = yatirim_tl / gelir_tl

print("=== FİNANSAL ANALİZ ===")
print(f"Arazi: {arazi_hektar} hektar")
print(f"Kurulu güç: {kurulu_guc_mw:.0f} MW")
print(f"Yıllık üretim: {uretim_kwh/1000:,.0f} MWh")
print(f"Yıllık gelir: {gelir_tl:,.0f} TL")
print(f"Yatırım maliyeti: ${yatirim_dolar:,.0f} / {yatirim_tl:,.0f} TL")
print(f"Geri ödeme süresi: {geri_odeme:.1f} yıl")

=== FİNANSAL ANALİZ ===
Arazi: 100 hektar
Kurulu güç: 50 MW
Yıllık üretim: 70,000 MWh
Yıllık gelir: 293,998,740 TL
Yatırım maliyeti: $60,000,000 / 2,280,000,000 TL
Geri ödeme süresi: 7.8 yıl


In [12]:
print("=" * 45)
print("       GEOHAN SOLAR-INTELLIGENCE RAPORU")
print("=" * 45)
print(f"\n📍 BÖLGE: Konya-Ankara")
print(f"📐 KOORDİNAT: {lat}°N, {lon}°E")
print()
print("─" * 45)
print("🌍 ARAZİ ANALİZİ")
print("─" * 45)
print(f"  Ortalama eğim:     {egim_derece.mean():.1f}°")
print(f"  Yıllık GHI:        {yillik_ghi:.0f} kWh/m²/yıl")
print()
print("─" * 45)
print("⚡ ENERJİ POTANSİYELİ (100 hektar)")
print("─" * 45)
print(f"  Kurulu güç:        {kurulu_guc_mw:.0f} MW")
print(f"  Yıllık üretim:     {uretim_kwh/1000:,.0f} MWh")
print()
print("─" * 45)
print("💰 FİNANSAL ÖZET")
print("─" * 45)
print(f"  Yatırım:           ${yatirim_dolar:,.0f}")
print(f"  Yıllık gelir:      {gelir_tl/1_000_000:.1f} milyon TL")
print(f"  Geri ödeme:        {geri_odeme:.1f} yıl")
print()
print("─" * 45)
print("🏆 SOLAR-INTELLIGENCE SKORU")
print("─" * 45)
print(f"  Eğim skoru:        {egim_skoru}/100")
print(f"  GHI skoru:         {ghi_skoru}/100")
print(f"  TOPLAM:            {solar_skor:.0f}/100 ✅")
print("=" * 45)
print("         geohanmaps.com")
print("=" * 45)

       GEOHAN SOLAR-INTELLIGENCE RAPORU

📍 BÖLGE: Konya-Ankara
📐 KOORDİNAT: 38.5°N, 32.5°E

─────────────────────────────────────────────
🌍 ARAZİ ANALİZİ
─────────────────────────────────────────────
  Ortalama eğim:     3.5°
  Yıllık GHI:        1750 kWh/m²/yıl

─────────────────────────────────────────────
⚡ ENERJİ POTANSİYELİ (100 hektar)
─────────────────────────────────────────────
  Kurulu güç:        50 MW
  Yıllık üretim:     70,000 MWh

─────────────────────────────────────────────
💰 FİNANSAL ÖZET
─────────────────────────────────────────────
  Yatırım:           $60,000,000
  Yıllık gelir:      294.0 milyon TL
  Geri ödeme:        7.8 yıl

─────────────────────────────────────────────
🏆 SOLAR-INTELLIGENCE SKORU
─────────────────────────────────────────────
  Eğim skoru:        100/100
  GHI skoru:         80/100
  TOPLAM:            88/100 ✅
         geohanmaps.com


In [13]:
import requests

# TCMB güncel kur
tcmb_url = "https://www.tcmb.gov.tr/kurlar/today.xml"
response = requests.get(tcmb_url)

import xml.etree.ElementTree as ET
root = ET.fromstring(response.content)

usd_tl = None
for currency in root.findall('Currency'):
    if currency.get('Kod') == 'USD':
        usd_tl = float(currency.find('ForexSelling').text.replace(',', '.'))
        break

print(f"Güncel USD/TL kuru: {usd_tl}")

Güncel USD/TL kuru: 45.0202


In [15]:
# Güncel parametreler
kwh_fiyat_tl = 4.20  # PTF ortalama tahmini - epias.gov.tr'den güncellenmeli
yatirim_mw_dolar = 900_000  # Türkiye 2024 sektör ortalaması
performans = 0.80

# Hesaplar
yatirim_dolar = kurulu_guc_mw * yatirim_mw_dolar
yatirim_tl = yatirim_dolar * usd_tl
gelir_tl = uretim_kwh * kwh_fiyat_tl
geri_odeme = yatirim_tl / gelir_tl

print("=== FİNANSAL ANALİZ (Güncel Kur) ===")
print(f"USD/TL kuru:       {usd_tl}")
print(f"Yatırım:           ${yatirim_dolar:,.0f} / {yatirim_tl:,.0f} TL")
print(f"Yıllık gelir:      {gelir_tl/1_000_000:.1f} milyon TL")
print(f"Geri ödeme:        {geri_odeme:.1f} yıl")

=== FİNANSAL ANALİZ (Güncel Kur) ===
USD/TL kuru:       45.0202
Yatırım:           $45,000,000 / 2,025,909,000 TL
Yıllık gelir:      294.0 milyon TL
Geri ödeme:        6.9 yıl


In [16]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.units import cm

print("ReportLab hazır!")

ReportLab hazır!


In [17]:
doc = SimpleDocTemplate(
    "GeoHan_Solar_Raporu.pdf",
    pagesize=A4,
    rightMargin=2*cm,
    leftMargin=2*cm,
    topMargin=2*cm,
    bottomMargin=2*cm
)

styles = getSampleStyleSheet()
elements = []

# Başlık
elements.append(Paragraph("GEOHAN SOLAR-INTELLIGENCE RAPORU", styles['Title']))
elements.append(Spacer(1, 0.5*cm))

# Bölge bilgisi
elements.append(Paragraph(f"Bölge: Konya-Ankara", styles['Heading2']))
elements.append(Paragraph(f"Koordinat: {lat}°N, {lon}°E", styles['Normal']))
elements.append(Spacer(1, 0.5*cm))

# Arazi analizi tablosu
elements.append(Paragraph("Arazi Analizi", styles['Heading2']))
arazi_data = [
    ["Parametre", "Değer"],
    ["Ortalama Eğim", f"{egim_derece.mean():.1f}°"],
    ["Yıllık GHI", f"{yillik_ghi:.0f} kWh/m²/yıl"],
    ["Uygun Alan Oranı", "%82.4"],
]
arazi_tablo = Table(arazi_data, colWidths=[8*cm, 8*cm])
arazi_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F2F3F4')]),
]))
elements.append(arazi_tablo)
elements.append(Spacer(1, 0.5*cm))

# Finansal tablo
elements.append(Paragraph("Finansal Analiz (100 hektar)", styles['Heading2']))
finans_data = [
    ["Parametre", "Değer"],
    ["Kurulu Güç", f"{kurulu_guc_mw:.0f} MW"],
    ["Yıllık Üretim", f"{uretim_kwh/1000:,.0f} MWh"],
    ["Yatırım Maliyeti", f"${yatirim_dolar:,.0f}"],
    ["Yıllık Gelir", f"{gelir_tl/1_000_000:.1f} milyon TL"],
    ["Geri Ödeme Süresi", f"{geri_odeme:.1f} yıl"],
    ["USD/TL Kuru", f"{usd_tl} (TCMB)"],
]
finans_tablo = Table(finans_data, colWidths=[8*cm, 8*cm])
finans_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F2F3F4')]),
]))
elements.append(finans_tablo)
elements.append(Spacer(1, 0.5*cm))

# Skor
elements.append(Paragraph("Solar-Intelligence Skoru", styles['Heading2']))
skor_data = [
    ["Kriter", "Skor"],
    ["Eğim", f"{egim_skoru}/100"],
    ["GHI", f"{ghi_skoru}/100"],
    ["TOPLAM", f"{solar_skor:.0f}/100"],
]
skor_tablo = Table(skor_data, colWidths=[8*cm, 8*cm])
skor_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor('#27AE60')),
    ('TEXTCOLOR', (0,-1), (-1,-1), colors.white),
    ('FONTNAME', (0,-1), (-1,-1), 'Helvetica-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
]))
elements.append(skor_tablo)
elements.append(Spacer(1, 1*cm))

# Footer
elements.append(Paragraph("⚠️ Yatırım maliyeti tahminidir. Gerçek teklif için müteahhidinize danışınız.", styles['Normal']))
elements.append(Spacer(1, 0.3*cm))
elements.append(Paragraph("geohanmaps.com", styles['Normal']))

# PDF oluştur
doc.build(elements)
print("PDF oluşturuldu: GeoHan_Solar_Raporu.pdf")

PDF oluşturuldu: GeoHan_Solar_Raporu.pdf


In [18]:
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import os

# Windows sistem fontunu kullan
font_path = "C:/Windows/Fonts/arial.ttf"
font_bold_path = "C:/Windows/Fonts/arialbd.ttf"

pdfmetrics.registerFont(TTFont('Arial', font_path))
pdfmetrics.registerFont(TTFont('Arial-Bold', font_bold_path))

print("Font kaydedildi!")

Font kaydedildi!


In [19]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle

# Arial fontlu stiller
baslik = ParagraphStyle('baslik', fontName='Arial-Bold', fontSize=16, alignment=1, spaceAfter=10)
baslik2 = ParagraphStyle('baslik2', fontName='Arial-Bold', fontSize=12, spaceAfter=6)
normal = ParagraphStyle('normal', fontName='Arial', fontSize=10, spaceAfter=4)
footer = ParagraphStyle('footer', fontName='Arial', fontSize=8, textColor=colors.grey)

doc = SimpleDocTemplate(
    "GeoHan_Solar_Raporu.pdf",
    pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm,
    topMargin=2*cm, bottomMargin=2*cm
)

elements = []

elements.append(Paragraph("GEOHAN SOLAR-INTELLIGENCE RAPORU", baslik))
elements.append(Spacer(1, 0.3*cm))
elements.append(Paragraph(f"Bölge: Konya-Ankara", baslik2))
elements.append(Paragraph(f"Koordinat: {lat}°N, {lon}°E", normal))
elements.append(Spacer(1, 0.5*cm))

elements.append(Paragraph("Arazi Analizi", baslik2))
arazi_data = [
    ["Parametre", "Değer"],
    ["Ortalama Eğim", f"{egim_derece.mean():.1f}°"],
    ["Yıllık GHI", f"{yillik_ghi:.0f} kWh/m²/yıl"],
    ["Uygun Alan Oranı", "%82.4"],
]
arazi_tablo = Table(arazi_data, colWidths=[8*cm, 8*cm])
arazi_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,-1), 'Arial'),
    ('FONTNAME', (0,0), (-1,0), 'Arial-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F2F3F4')]),
]))
elements.append(arazi_tablo)
elements.append(Spacer(1, 0.5*cm))

elements.append(Paragraph("Finansal Analiz (100 hektar)", baslik2))
finans_data = [
    ["Parametre", "Değer"],
    ["Kurulu Güç", f"{kurulu_guc_mw:.0f} MW"],
    ["Yıllık Üretim", f"{uretim_kwh/1000:,.0f} MWh"],
    ["Yatırım Maliyeti", f"${yatirim_dolar:,.0f}"],
    ["Yıllık Gelir", f"{gelir_tl/1_000_000:.1f} milyon TL"],
    ["Geri Ödeme Süresi", f"{geri_odeme:.1f} yıl"],
    ["USD/TL Kuru", f"{usd_tl} (TCMB)"],
]
finans_tablo = Table(finans_data, colWidths=[8*cm, 8*cm])
finans_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,-1), 'Arial'),
    ('FONTNAME', (0,0), (-1,0), 'Arial-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F2F3F4')]),
]))
elements.append(finans_tablo)
elements.append(Spacer(1, 0.5*cm))

elements.append(Paragraph("Solar-Intelligence Skoru", baslik2))
skor_data = [
    ["Kriter", "Skor"],
    ["Eğim", f"{egim_skoru}/100"],
    ["GHI", f"{ghi_skoru}/100"],
    ["TOPLAM", f"{solar_skor:.0f}/100"],
]
skor_tablo = Table(skor_data, colWidths=[8*cm, 8*cm])
skor_tablo.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2C3E50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,-1), 'Arial'),
    ('FONTNAME', (0,0), (-1,0), 'Arial-Bold'),
    ('FONTNAME', (0,-1), (-1,-1), 'Arial-Bold'),
    ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor('#27AE60')),
    ('TEXTCOLOR', (0,-1), (-1,-1), colors.white),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
]))
elements.append(skor_tablo)
elements.append(Spacer(1, 1*cm))

elements.append(Paragraph("⚠️ Yatırım maliyeti tahminidir. Gerçek teklif için müteahhidinize danışınız.", footer))
elements.append(Spacer(1, 0.3*cm))
elements.append(Paragraph("geohanmaps.com", footer))

doc.build(elements)
print("PDF oluşturuldu!")

PDF oluşturuldu!
